# Lab 06｜只继承合适位置以前的历史

旧事故的模块背景仍能用于新调查，旧根因却不能直接沿用。先给父 Thread 留下模块背景，再追加旧事故结论，然后从背景所在 Turn 分岔，检查新任务取得了哪些历史、排除了哪些历史。

## 核心代码：以指定 Turn 为边界创建历史分支

> 本讲重点看下面这 5 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**公共准备，第 3 个单元格的 `fork_at_background()`，第 90 至 93 行：**

```python
def fork_at_background():
    response = fork_client.thread_fork(parent.id, {"lastTurnId":base.id,
        "cwd":str(WORK), "sandbox":"read-only", "approvalPolicy":"never"})
    return Thread(fork_client,response.thread.id)
```

`thread_fork()` 创建分支，`lastTurnId=base.id` 指定包含背景 Turn 的继承边界；后面的旧事故结论不在这段继承历史里。返回的 `Thread` 使用新分支编号。

**第 2 节，第 7 个单元格，第 1 行：**

```python
branch_a = fork_at_background()
```

这一行调用上面的函数，得到后续调查使用的子 Thread。父子仍共用 `cwd`，历史分支不等于文件隔离。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `DEEPSEEK_API_KEY`；未设置时弹出隐藏输入框。模型通过原生 Responses 使用 DeepSeek Flash，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、请求观察和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(package + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex",))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("DEEPSEEK_API_KEY", "").strip():
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DEEPSEEK_API_KEY：").strip()
if "lab" in globals() and not globals()["lab"]._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "06-thread-fork")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))



WORK = lab.work
from openai_codex.client import CodexClient
from openai_codex.api import Thread
fork_client = lab.manage_client(CodexClient(config=lab.config))
fork_client.start()
fork_client.initialize()

def fork_at_background():
    response = fork_client.thread_fork(parent.id, {"lastTurnId":base.id,
        "cwd":str(WORK), "sandbox":"read-only", "approvalPolicy":"never"})
    return Thread(fork_client,response.thread.id)

print("公共准备完成。")


## 1. 把共同背景与旧结论分开放｜必读必跑

第一轮放跨事故可用的模块背景，第二轮才放旧事故结论。接下来选择第一轮作为分岔边界，看看哪些内容被继承。

基础观察只用一条父 Thread：Turn 1 放背景、Turn 2 放旧结论，再从 Turn 1 分岔。可选反例把旧根因也放进 Turn 1；核对它是否随分支继承，不以模型回答声称“不会沿用”代替输入检查。


In [ ]:
# 可选反例：改为 True 后从本讲公共准备重新运行；仍只创建一条父 Thread。
# 若旧根因进入 Turn 1，它也会被 Turn 1 的分支继承。
RUN_CONTAMINATED_BACKGROUND = False
parent = lab.thread()
background = "历史支付模块使用独立部署脚本，日志入口为 legacy_payment_logs。这些是跨事故线索，新任务仍须重新核对。"
old_cause = "上一场事故中连接池上限被调低，恢复后症状消失。这只属于旧事故。"
background_turn = background + ("\n" + old_cause if RUN_CONTAMINATED_BACKGROUND else "")
base = run(parent, background_turn + "仅确认，不调用工具。")
old_result = run(parent, old_cause + "仅确认，不调用工具。")
parent_before = parent.read(include_turns=True).model_dump(mode="json",by_alias=True)
table(["父 Thread 中的位置", "放入的内容"],[("Turn 1 / 边界内：背景",background_turn),("Turn 2 / 边界后：旧结论",old_cause)])


## 2. 从背景 Turn 分岔｜必读必跑

固定版本的高层接口未暴露 `last_turn_id`，辅助函数使用同一 SDK 的底层 `lastTurnId`；这个边界包含所选 Turn。

下面先建一个子 Thread。对照实际请求与历史，检查“父：背景＋旧结论；子：背景＋新任务”。第二个分支只作可选扩展。


In [ ]:
branch_a = fork_at_background()
new_task = "新事故 A：支付延迟再次上升，请依据继承的模块背景说明下一步调查方向，不沿用旧事故根因，不调用工具。"
a_start = len(lab.requests)
a_result = run(branch_a,new_task)
a_request = lab.request_text(a_start)
a_history = branch_a.read(include_turns=True).model_dump(mode="json",by_alias=True)
parent_after = parent.read(include_turns=True).model_dump(mode="json",by_alias=True)
parent_text = json.dumps(parent_after,ensure_ascii=False)
table(["Thread", "共同背景", "旧事故结论", "新任务"], [
    (name, *["有" if value in text else "无" for value in (background,old_cause,new_task)])
    for name,text in [("父：原历史",parent_text),("子：实际模型请求",a_request)]
])
display(Markdown("### 子 Thread 的实际回答\n" + a_result.final_response))
details("扩展：分支来源与父历史核对",json.dumps({
    "子分支是否标明父 Thread":a_history["thread"].get("forkedFromId")==parent.id,
    "父历史是否保持原样":parent_before["thread"]["turns"]==parent_after["thread"]["turns"],
},ensure_ascii=False))

RUN_SECOND_BRANCH = False  # 扩展/可选
if RUN_SECOND_BRANCH:
    branch_b = fork_at_background()
    b_start = len(lab.requests)
    b_result = run(branch_b,"新事故 B：下游响应变慢，依据继承的背景说明调查方向，不调用工具。")
    table(["第二个分支的首次请求", "实际内容"], [
        ("共同背景","有" if background in lab.request_text(b_start) else "无"),
        ("A 的新增任务","有" if new_task in lab.request_text(b_start) else "无"),
    ])


## 3. 共用工作目录的文件观察｜扩展/可选，默认不运行

基础实验到第 2 节已完成，读完输入对照即可执行最后的清理格。

若将下格 `RUN_SHARED_FILE` 改为 `True`，再请子 Thread 写一份便笺、父 Thread 读取。它额外调用两次模型，用来区分历史分支与文件快照：父并未继承子对话，却能通过新的工具调用读到共享目录中的文件。


In [ ]:
RUN_SHARED_FILE = False
if RUN_SHARED_FILE:
    shared_writer = run(branch_a,
        "把‘支付延迟再次上升，原因仍待核对’写入 shared-note.txt，只创建这份便笺，不改其他文件。",
        sandbox=Sandbox.workspace_write)
    shared_reader = run(parent,"读取 shared-note.txt，说明你现在从便笺得到什么新信息，不修改文件。")
    table(["Thread", "共享目录中的实际动作"],[("子 Thread",shared_writer.final_response),("父 Thread",shared_reader.final_response)])
    events(shared_reader)


## 从本次观察带走什么

先证明所选历史前缀含背景、不含后来的旧结论，再检查父历史未受子任务污染。旧结论若已在边界内，分叉仍会继承；父子共享目录则仍能交换文件。历史、文件和外部资源的隔离需要分别设计。


In [ ]:
# 继续观察完成后关闭本讲所有连接。
lab.close()
